In [30]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(url, names=cols, na_values='?').dropna()
df['target'] = (df['target'] > 0).astype(int)

df.head(10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
5,56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
6,62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,1
7,57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0
8,63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,1
9,53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1


In [31]:
!pip uninstall flaml -y
!pip install flaml[automl]

Found existing installation: FLAML 2.6.0
Uninstalling FLAML-2.6.0:
  Successfully uninstalled FLAML-2.6.0
  Using cached flaml-2.6.0-py3-none-any.whl.metadata (12 kB)
Using cached flaml-2.6.0-py3-none-any.whl (349 kB)


In [32]:
#H2O AutoML на данных UCI Heart Disease
import warnings
warnings.filterwarnings('ignore')
!pip install h2o -q
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import h2o
from h2o.automl import H2OAutoML
import time
import flaml
from flaml import AutoML

In [33]:
# Загрузка данных
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(url, names=columns, na_values='?')
df.dropna(inplace=True)
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {X_train.shape}, Тестовая: {X_test.shape}")

Обучающая выборка: (237, 13), Тестовая: (60, 13)


In [43]:
h2o.init(max_mem_size="2G", nthreads=2, verbose=False)
train_h2o = h2o.H2OFrame(pd.concat([X_train, y_train], axis=1))
test_h2o = h2o.H2OFrame(pd.concat([X_test, y_test], axis=1))
train_h2o['target'] = train_h2o['target'].asfactor()
test_h2o['target'] = test_h2o['target'].asfactor()

time_budget_sec = 120  

aml = H2OAutoML(max_runtime_secs=time_budget_sec, seed=42, verbosity=None)
start = time.time()
aml.train(x=X_train.columns.tolist(), y='target', training_frame=train_h2o)
h2o_time = time.time() - start

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


In [44]:
# Оценка
perf = aml.leader.model_performance(test_h2o)
preds = aml.leader.predict(test_h2o)
probs = preds['p1'].as_data_frame().values.flatten()
classes = preds['predict'].as_data_frame().values.flatten()

roc = roc_auc_score(y_test, probs)
acc = accuracy_score(y_test, classes)
f1 = f1_score(y_test, classes)

print("\n========== РЕЗУЛЬТАТЫ H2O AutoML ==========")
print(f"ROC-AUC: {roc:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Время обучения: {h2o_time:.1f} сек")

gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%

========== РЕЗУЛЬТАТЫ H2O AutoML ==========
ROC-AUC: 0.9387
Accuracy: 0.8833
F1-score: 0.8627
Время обучения: 173.2 сек


In [45]:
# Важность признаков
varimp = aml.leader.varimp(use_pandas=True)
print("\nВажность признаков (H2O) – top 5:")
print(varimp.head(5))

h2o.cluster().shutdown()

globals()['h2o_roc'] = roc
globals()['h2o_acc'] = acc
globals()['h2o_f1'] = f1
globals()['h2o_time'] = h2o_time


Важность признаков (H2O) – top 5:
  variable  relative_importance  scaled_importance  percentage
0       ca            52.523605           1.000000    0.208161
1     thal            37.460075           0.713205    0.148461
2       cp            29.431545           0.560349    0.116643
3  oldpeak            24.083067           0.458519    0.095446
4      age            23.497042           0.447362    0.093123


In [46]:
# ========== FLAML на данных UCI Heart Disease ==========
!pip install flaml -q

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.inspection import permutation_importance
from flaml import AutoML
import time

In [47]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(url, names=columns, na_values='?')
df.dropna(inplace=True)
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [48]:
# Обучение FLAML
automl = AutoML()
start = time.time()
automl.fit(X_train, y_train, task="classification", time_budget=120, metric='roc_auc')
flaml_time = time.time() - start

[flaml.automl.logger: 05-13 23:27:06] {2375} INFO - task = classification
[flaml.automl.logger: 05-13 23:27:06] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 05-13 23:27:06] {2489} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 05-13 23:27:06] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 05-13 23:27:06] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 05-13 23:27:06] {3046} INFO - Estimated sufficient time budget=454s. Estimated necessary time budget=11s.
[flaml.automl.logger: 05-13 23:27:06] {3097} INFO -  at 0.1s,	estimator lgbm's best error=1.8411e-01,	best estimator lgbm's best error=1.8411e-01
[flaml.automl.logger: 05-13 23:27:06] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-13 23:27:06] {3097} INFO -  at 0.1s,	estimator lgbm's best error=1.7012e-01,	best estimator lgbm's best error=1.7012e-01
[flaml.automl

In [49]:
# Предсказания
y_proba = automl.predict_proba(X_test)[:, 1]
y_pred = automl.predict(X_test)
roc = roc_auc_score(y_test, y_proba)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n========== РЕЗУЛЬТАТЫ FLAML ==========")
print(f"ROC-AUC: {roc:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Время обучения: {flaml_time:.1f} сек")


========== РЕЗУЛЬТАТЫ FLAML ==========
ROC-AUC: 0.9306
Accuracy: 0.8667
F1-score: 0.8462
Время обучения: 120.0 сек


In [50]:
# Важность признаков
print("\nВычисление важности признаков:")
perm_imp = permutation_importance(automl, X_test, y_test, n_repeats=5, random_state=42, n_jobs=1)
print("\nВажность признаков (top-5):")
imp_df = pd.DataFrame({'feature': X.columns, 'importance': perm_imp.importances_mean})
imp_df = imp_df.sort_values('importance', ascending=False)
print(imp_df.head(5))

globals()['flaml_roc'] = roc
globals()['flaml_acc'] = acc
globals()['flaml_f1'] = f1
globals()['flaml_time'] = flaml_time


Вычисление важности признаков:

Важность признаков (top-5):
    feature  importance
12     thal    0.086667
2        cp    0.060000
9   oldpeak    0.050000
11       ca    0.036667
1       sex    0.026667


In [51]:
# ========== Сравнительная таблица H2O vs FLAML ==========
import pandas as pd

try:
    comparison = pd.DataFrame({
        'Фреймворк': ['H2O AutoML', 'FLAML'],
        'ROC-AUC': [h2o_roc, flaml_roc],
        'Accuracy': [h2o_acc, flaml_acc],
        'F1-score': [h2o_f1, flaml_f1],
        'Время (сек)': [h2o_time, flaml_time]
    })
    print("\n========== ИТОГОВОЕ СРАВНЕНИЕ ==========")
    print(comparison.round(4))

    print("\n=========== ВЫВОД ===========")
    if h2o_roc > flaml_roc:
        print("H2O AutoML показал более высокое качество (ROC-AUC).")
    else:
        print("FLAML показал более высокое качество (ROC-AUC).")

    print(f"Разница во времени: {abs(h2o_time - flaml_time):.1f} сек")
    print("Интерпретируемость лучше у H2O (встроенная varimp против permutation importance).")
    print("Рекомендация для задачи выявления профилей риска ССЗ: H2O AutoML.")

except NameError:
    print("Ошибка: не найдены метрики. Запустите сначала блоки H2O и FLAML.")
    print("Если вы перезапускали сессию, запустите всё заново по порядку.")


========== ИТОГОВОЕ СРАВНЕНИЕ ==========
    Фреймворк  ROC-AUC  Accuracy  F1-score  Время (сек)
0  H2O AutoML   0.9387    0.8833    0.8627     173.1534
1       FLAML   0.9306    0.8667    0.8462     120.0232

=========== ВЫВОД ===========
H2O AutoML показал более высокое качество (ROC-AUC).
Разница во времени: 53.1 сек
Интерпретируемость лучше у H2O (встроенная varimp против permutation importance).
Рекомендация для задачи выявления профилей риска ССЗ: H2O AutoML.
